                ┌────────────────────────────┐
                │        Raw Dataset         │
                │  - subject (text)          │
                │  - sender_domain (category)│
                │  - num_links, num_words    │
                │  - received_date (string)  │
                │  - label (spam/not_spam)   │
                └──────────────┬─────────────┘
                               │
                               ▼
                ┌────────────────────────────┐
                │     Feature Engineering    │
                │  - Convert date → datetime │
                │  - Extract day_of_week     │
                │  - Extract month           │
                │  - Drop raw date column    │
                └──────────────┬─────────────┘
                               │
                               ▼
                ┌────────────────────────────┐
                │      Train/Test Split      │
                │  X = features              │
                │  y = label                 │
                │  LabelEncoder(y)           │
                │  70% train / 30% test      │
                └──────────────┬─────────────┘
                               │
                               ▼
        ┌──────────────────────────────────────────────────┐
        │              ColumnTransformer                   │
        │                                                  │
        │   ┌───────────────┐   ┌──────────────────┐       │
        │   │  Text (TF‑IDF)│   │Categorical (OHE) │       │
        │   │  subject      │   │sender_domain     │       │
        │   └───────────────┘   └──────────────────┘       │
        │                                                  │
        │   ┌──────────────────────────────┐               │
        │   │ Numeric (Imputer: median)    │               │
        │   │ num_links, num_words,        │               │
        │   │ day_of_week, month           │               │
        │   └──────────────────────────────┘               │
        └───────────────────────┬──────────────────────────┘
                                │
                                ▼
                ┌────────────────────────────┐
                │          Pipeline          │
                │  preprocess → (model later)│
                └──────────────┬─────────────┘
                               │
                               ▼
                ┌────────────────────────────┐
                │   ML‑Ready Feature Matrix  │
                │  - Sparse matrix           │
                │  - All numeric             │
                │  - Combined features       │
                └────────────────────────────┘


In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, LabelEncoder
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer

# -----------------------------
# 1. Load dataset
# -----------------------------
data = {
    "subject": [
        "Win a free vacation",
        "Your receipt from Bookstore",
        "Limited time offer",
        "Meeting agenda",
        "Update your account info",
        "Family photos inside",
        "Exclusive promo just for you",
        "Weekly project status"
    ],
    "label": [
        "spam",
        "not_spam",
        "spam",
        "not_spam",
        "spam",
        "not_spam",
        "spam",
        "not_spam"
    ],
    "sender_domain": [
        "promos.biz",
        "gmail.com",
        "offers.mail",
        "company.com",
        "alerts.bank",
        "yahoo.com",
        "promos.biz",
        "company.com"
    ],
    "num_links": [5, 0, 7, 1, np.nan, 0, 4, 1],
    "num_words": [40, 120, 35, 80, 60, 100, 50, 90],
    "received_date": [
        "2024-01-15",
        "2024-01-18",
        "2024-02-02",
        "2024-02-10",
        "2024-03-05",
        "2024-03-08",
        "2024-03-20",
        "2024-04-01"
    ]
}

df = pd.DataFrame(data)
df

,subject,label,sender_domain,num_links,num_words,received_date
0,Win a free vacation,spam,promos.biz,5.0,40,2024-01-15
1,Your receipt from Bookstore,not_spam,gmail.com,0.0,120,2024-01-18
2,Limited time offer,spam,offers.mail,7.0,35,2024-02-02
3,Meeting agenda,not_spam,company.com,1.0,80,2024-02-10
4,Update your account info,spam,alerts.bank,NaN,60,2024-03-05
5,Family photos inside,not_spam,yahoo.com,0.0,100,2024-03-08
6,Exclusive promo just for you,spam,promos.biz,4.0,50,2024-03-20
7,Weekly project status,not_spam,company.com,1.0,90,2024-04-01


In [2]:
# -----------------------------
# 2. Feature engineering
# -----------------------------
df["received_date"] = pd.to_datetime(df["received_date"])
df["day_of_week"] = df["received_date"].dt.dayofweek
df["month"] = df["received_date"].dt.month

# Drop raw date column (optional)
df = df.drop(columns=["received_date"])
df

,subject,label,sender_domain,num_links,num_words,day_of_week,month
0,Win a free vacation,spam,promos.biz,5.0,40,0,1
1,Your receipt from Bookstore,not_spam,gmail.com,0.0,120,3,1
2,Limited time offer,spam,offers.mail,7.0,35,4,2
3,Meeting agenda,not_spam,company.com,1.0,80,5,2
4,Update your account info,spam,alerts.bank,NaN,60,1,3
5,Family photos inside,not_spam,yahoo.com,0.0,100,4,3
6,Exclusive promo just for you,spam,promos.biz,4.0,50,2,3
7,Weekly project status,not_spam,company.com,1.0,90,0,4


In [3]:
# -----------------------------
# 3. Train/test split
# -----------------------------
X = df.drop(columns=["label"])
y = df["label"]

# Encode label
label_encoder = LabelEncoder()
y_encoded = label_encoder.fit_transform(y)

X_train, X_test, y_train, y_test = train_test_split(
    X, y_encoded, test_size=0.25, random_state=42
)

# -----------------------------
# 4. Build preprocessing pipeline
# -----------------------------
text_features = ["subject"]
categorical_features = ["sender_domain"]
numeric_features = ["num_links", "num_words", "day_of_week", "month"]

preprocessor = ColumnTransformer(
    transformers=[
        ("text", TfidfVectorizer(), "subject"),
        ("cat", OneHotEncoder(handle_unknown="ignore"), categorical_features),
        ("num", SimpleImputer(strategy="median"), numeric_features)
    ]
)

# -----------------------------
# 5. Final pipeline (ready for model)
# -----------------------------
pipeline = Pipeline(steps=[
    ("preprocess", preprocessor)
    # Add model here, e.g.:
    # ("model", LogisticRegression())
])

# Fit preprocessing only (no model yet)
pipeline.fit(X_train)

# Transform data
X_train_prepared = pipeline.transform(X_train)
X_test_prepared = pipeline.transform(X_test)

print("Training matrix shape:", X_train_prepared.shape)
print("Test matrix shape:", X_test_prepared.shape)


Training matrix shape: (6, 28)
Test matrix shape: (2, 28)


In [4]:
pd.DataFrame(data)

,subject,label,sender_domain,num_links,num_words,received_date
0,Win a free vacation,spam,promos.biz,5.0,40,2024-01-15
1,Your receipt from Bookstore,not_spam,gmail.com,0.0,120,2024-01-18
2,Limited time offer,spam,offers.mail,7.0,35,2024-02-02
3,Meeting agenda,not_spam,company.com,1.0,80,2024-02-10
4,Update your account info,spam,alerts.bank,NaN,60,2024-03-05
5,Family photos inside,not_spam,yahoo.com,0.0,100,2024-03-08
6,Exclusive promo just for you,spam,promos.biz,4.0,50,2024-03-20
7,Weekly project status,not_spam,company.com,1.0,90,2024-04-01


In [5]:
pd.DataFrame(X_train_prepared.toarray())

,0,1,2,3,4,5,6,7,8,9,...,18,19,20,21,22,23,24,25,26,27
0,0.0,0.000000,0.000000,0.000000,0.57735,0.0,0.000000,0.00000,0.000000,0.00000,...,0.000000,0.0,0.0,0.0,0.0,1.0,5.0,40.0,0.0,1.0
1,0.0,0.000000,0.000000,0.000000,0.00000,0.0,0.000000,0.00000,0.000000,0.00000,...,0.000000,0.0,0.0,1.0,0.0,0.0,1.0,90.0,0.0,4.0
2,0.0,0.000000,0.000000,0.000000,0.00000,0.0,0.000000,0.57735,0.000000,0.57735,...,0.000000,0.0,0.0,0.0,1.0,0.0,7.0,35.0,4.0,2.0
3,0.5,0.000000,0.000000,0.000000,0.00000,0.5,0.000000,0.00000,0.000000,0.00000,...,0.000000,0.5,1.0,0.0,0.0,0.0,4.0,60.0,1.0,3.0
4,0.0,0.707107,0.000000,0.000000,0.00000,0.0,0.000000,0.00000,0.707107,0.00000,...,0.000000,0.0,0.0,1.0,0.0,0.0,1.0,80.0,5.0,2.0
5,0.0,0.000000,0.447214,0.447214,0.00000,0.0,0.447214,0.00000,0.000000,0.00000,...,0.447214,0.0,0.0,0.0,0.0,1.0,4.0,50.0,2.0,3.0


In [6]:
y_train

array([1, 0, 1, 1, 0, 1])

In [7]:
pd.DataFrame(X_test_prepared.toarray())

,0,1,2,3,4,5,6,7,8,9,...,18,19,20,21,22,23,24,25,26,27
0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,1.0,0.0,0.0,0.0,0.0,0.0,120.0,3.0,1.0
1,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,0.0,...,0.0,0.0,0.0,0.0,0.0,0.0,0.0,100.0,4.0,3.0


In [8]:
y_test

array([0, 0])

In [9]:
# Vectorizer example
print(df['subject'])
print(df['subject'].shape[0])

0             Win a free vacation
1     Your receipt from Bookstore
2              Limited time offer
3                  Meeting agenda
4        Update your account info
5            Family photos inside
6    Exclusive promo just for you
7           Weekly project status
Name: subject, dtype: object
8


In [10]:

# Create the vectorizer
vectorizer = TfidfVectorizer()

# Fit and transform the text column
X = vectorizer.fit_transform(df['subject'])

print(X.toarray())

X.toarray().shape


[[0.         0.         0.         0.         0.         0.
  0.57735027 0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.57735027 0.         0.57735027
  0.         0.        ]
 [0.         0.         0.51970849 0.         0.         0.
  0.         0.51970849 0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.51970849
  0.         0.         0.         0.         0.         0.
  0.         0.43555627]
 [0.         0.         0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.57735027
  0.         0.57735027 0.         0.         0.         0.
  0.         0.57735027 0.         0.         0.         0.
  0.         0.        ]
 [0.         0.70710678 0.         0.         0.         0.
  0.         0.         0.         0.         0.         0.
  0.70710678 0.         0.         0.         0.         0.
 

(8, 26)

In [11]:
print(vectorizer.get_feature_names_out())

['account' 'agenda' 'bookstore' 'exclusive' 'family' 'for' 'free' 'from'
 'info' 'inside' 'just' 'limited' 'meeting' 'offer' 'photos' 'project'
 'promo' 'receipt' 'status' 'time' 'update' 'vacation' 'weekly' 'win'
 'you' 'your']
